# Analise de pontos apos saida da planta

Objetivo: encontrar pessoas com mais ocorrencias de marcacoes realizadas depois da ultima saida de portaria do dia.

Criterio de saida da planta: `descricao_coletor` contendo `PORT` e `dir = S`.

Criterio de ponto apos saida: registros posteriores a essa saida no mesmo dia em que `descricao_coletor` contem `PONTO` ou `planta = 999`.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ultimas_saidas_sem_planta_999.py").exists():
    NOTEBOOK_DIR = Path(r"C:\Users\A1CLNDO\OneDrive - Volkswagen AG\Trabalho\Projetos\3 - Ronda\Códigos Extras\Documentação\Analisando Saída de Colaborador")

PROJECT_ROOT = Path(r"C:\Users\A1CLNDO\OneDrive - Volkswagen AG\Trabalho\Projetos\3 - Ronda")
PARQUET_CANDIDATES = [
    PROJECT_ROOT / "Códigos Extras" / "Documentação" / "Análisando Ajustes Manuais" / "base" / "parquet" / "df_ronda.parquet",
    NOTEBOOK_DIR / "database" / "df_ronda.parquet",
    PROJECT_ROOT / "project" / "database" / "df_ronda.parquet",
]

PARQUET_PATH = next((p for p in PARQUET_CANDIDATES if p.exists()), None)
if PARQUET_PATH is None:
    raise FileNotFoundError("Nao encontrei df_ronda.parquet nos caminhos esperados.")

OUT_DIR = NOTEBOOK_DIR / "saida_pontos_apos_saida_port_s"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH, OUT_DIR

## Parametros

Ajuste as datas se quiser reduzir a janela. Por padrao, a analise usa todo o parquet encontrado.

In [ ]:
DATA_INICIO = None  # exemplo: "2025-01-01"
DATA_FIM = None     # exemplo: "2026-04-01"

# Use "Sim" para olhar apenas ajustes manuais, "Não" para base original, ou None para tudo.
FILTRO_MANUAL = None

TOP_N = 50
TOP_N_DETALHE = 20

PARQUET_SQL = PARQUET_PATH.as_posix()
schema_cols = pq.read_schema(PARQUET_PATH).names
manual_expr = 'CAST("registro_manual" AS VARCHAR)' if "registro_manual" in schema_cols else ('CAST("Manual" AS VARCHAR)' if "Manual" in schema_cols else "'Não'")

where_parts = ["cadastro IS NOT NULL", "data IS NOT NULL", "hora IS NOT NULL"]
if DATA_INICIO:
    where_parts.append(f"CAST(data AS DATE) >= DATE '{DATA_INICIO}'")
if DATA_FIM:
    where_parts.append(f"CAST(data AS DATE) <= DATE '{DATA_FIM}'")
if FILTRO_MANUAL:
    where_parts.append(f"{manual_expr} = '{FILTRO_MANUAL}'")

WHERE_SQL = " AND ".join(where_parts)
WHERE_SQL

## Construir a base de ocorrencias

Para cada pessoa e dia, o notebook encontra a ultima saida `PORT/S`. Depois, pega todos os registros posteriores no mesmo dia. O campo `eh_ponto_apos_saida` marca especificamente os eventos que parecem ponto (`PONTO` ou planta `999`).

In [ ]:
con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=4")

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW base_eventos AS
    SELECT
      CAST(cadastro AS VARCHAR) AS cadastro,
      CAST(nome AS VARCHAR) AS nome,
      CAST(filial AS VARCHAR) AS filial,
      CAST(data AS DATE) AS data_ref,
      CAST(CAST(data AS DATE) || ' ' || CAST(hora AS VARCHAR) AS TIMESTAMP) AS ts_evento,
      CAST(hora AS VARCHAR) AS hora,
      CAST(dir AS VARCHAR) AS dir,
      CAST(tipo AS VARCHAR) AS tipo,
      CAST(acesso AS VARCHAR) AS acesso,
      CAST(planta AS VARCHAR) AS planta,
      CAST(coletor AS VARCHAR) AS coletor,
      CAST(descricao_coletor AS VARCHAR) AS descricao_coletor,
      CAST(uso_acesso AS VARCHAR) AS uso_acesso,
      CAST(cargo AS VARCHAR) AS cargo,
      CAST(area AS VARCHAR) AS area,
      CAST(departamento AS VARCHAR) AS departamento,
      {manual_expr} AS manual,
      upper(trim(CAST(dir AS VARCHAR))) AS dir_norm,
      upper(coalesce(CAST(descricao_coletor AS VARCHAR), '')) AS desc_norm,
      trim(CAST(planta AS VARCHAR)) AS planta_norm
    FROM read_parquet('{PARQUET_SQL}')
    WHERE {WHERE_SQL}
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE saidas_portaria AS
    SELECT
      cadastro,
      data_ref,
      max(ts_evento) AS saida_port_ts
    FROM base_eventos
    WHERE dir_norm = 'S'
      AND desc_norm LIKE '%PORT%'
    GROUP BY cadastro, data_ref
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE saidas_portaria_detalhe AS
    SELECT * EXCLUDE(rn)
    FROM (
      SELECT
        b.*,
        row_number() OVER (PARTITION BY b.cadastro, b.data_ref ORDER BY b.ts_evento DESC) AS rn
      FROM base_eventos b
      JOIN saidas_portaria s
        ON b.cadastro = s.cadastro
       AND b.data_ref = s.data_ref
       AND b.ts_evento = s.saida_port_ts
      WHERE b.dir_norm = 'S'
        AND b.desc_norm LIKE '%PORT%'
    )
    WHERE rn = 1
    """
)

con.execute(
    """
    CREATE OR REPLACE TEMP TABLE ocorrencias_apos_saida AS
    SELECT
      b.cadastro,
      b.nome,
      b.filial,
      b.data_ref,
      s.saida_port_ts,
      sp.descricao_coletor AS saida_portaria_descricao,
      sp.planta AS saida_portaria_planta,
      b.ts_evento,
      b.hora,
      b.dir,
      b.tipo,
      b.acesso,
      b.planta,
      b.coletor,
      b.descricao_coletor,
      b.uso_acesso,
      b.cargo,
      b.area,
      b.departamento,
      b.manual,
      date_diff('minute', s.saida_port_ts, b.ts_evento) AS minutos_apos_saida,
      CASE WHEN b.desc_norm LIKE '%PONTO%' OR b.planta_norm = '999' THEN 1 ELSE 0 END AS eh_ponto_apos_saida
    FROM base_eventos b
    JOIN saidas_portaria s
      ON b.cadastro = s.cadastro
     AND b.data_ref = s.data_ref
    LEFT JOIN saidas_portaria_detalhe sp
      ON b.cadastro = sp.cadastro
     AND b.data_ref = sp.data_ref
    WHERE b.ts_evento > s.saida_port_ts
    """
)

totais = con.execute(
    """
    SELECT
      (SELECT COUNT(*) FROM base_eventos) AS linhas_base,
      (SELECT COUNT(*) FROM saidas_portaria) AS dias_com_saida_port_s,
      (SELECT COUNT(*) FROM ocorrencias_apos_saida) AS marcacoes_apos_saida,
      (SELECT SUM(eh_ponto_apos_saida) FROM ocorrencias_apos_saida) AS pontos_apos_saida
    """
).fetchdf()

totais

## Ranking de pessoas

O ranking principal prioriza `qtd_pontos_apos_saida`. A coluna `qtd_marcacoes_apos_saida` mostra qualquer marcacao posterior, mesmo quando nao parece ponto.

In [ ]:
ranking_pessoas = con.execute(
    """
    SELECT
      cadastro,
      any_value(nome) AS nome,
      any_value(cargo) AS cargo,
      any_value(area) AS area,
      any_value(departamento) AS departamento,
      manual,
      COUNT(*) AS qtd_marcacoes_apos_saida,
      SUM(eh_ponto_apos_saida) AS qtd_pontos_apos_saida,
      COUNT(DISTINCT data_ref) AS qtd_dias_com_ocorrencia,
      MIN(data_ref) AS primeira_ocorrencia,
      MAX(data_ref) AS ultima_ocorrencia,
      ROUND(AVG(minutos_apos_saida), 1) AS media_minutos_apos_saida,
      percentile_cont(0.5) WITHIN GROUP (ORDER BY minutos_apos_saida) AS mediana_minutos_apos_saida
    FROM ocorrencias_apos_saida
    GROUP BY cadastro, manual
    ORDER BY qtd_pontos_apos_saida DESC, qtd_marcacoes_apos_saida DESC, qtd_dias_com_ocorrencia DESC
    """
).fetchdf()

ranking_pessoas.head(TOP_N)

In [ ]:
resumo_por_manual = con.execute(
    """
    SELECT
      manual,
      COUNT(*) AS qtd_marcacoes_apos_saida,
      SUM(eh_ponto_apos_saida) AS qtd_pontos_apos_saida,
      COUNT(DISTINCT cadastro) AS pessoas,
      COUNT(DISTINCT data_ref) AS dias
    FROM ocorrencias_apos_saida
    GROUP BY manual
    ORDER BY manual
    """
).fetchdf()

resumo_por_cargo = con.execute(
    """
    SELECT
      cargo,
      manual,
      COUNT(*) AS qtd_marcacoes_apos_saida,
      SUM(eh_ponto_apos_saida) AS qtd_pontos_apos_saida,
      COUNT(DISTINCT cadastro) AS pessoas
    FROM ocorrencias_apos_saida
    GROUP BY cargo, manual
    ORDER BY qtd_pontos_apos_saida DESC
    """
).fetchdf()

display(resumo_por_manual)
display(resumo_por_cargo)

## Detalhe dos maiores casos

A tabela abaixo traz os eventos posteriores a saida de portaria para os maiores casos do ranking.

In [ ]:
top_cadastros = ranking_pessoas.head(TOP_N_DETALHE)[["cadastro", "manual"]].copy()
con.register("top_cadastros", top_cadastros)

detalhes_top = con.execute(
    """
    SELECT o.*
    FROM ocorrencias_apos_saida o
    JOIN top_cadastros t
      ON o.cadastro = t.cadastro
     AND o.manual = t.manual
    ORDER BY o.cadastro, o.data_ref, o.ts_evento
    """
).fetchdf()

detalhes_top.head(100)

## Visualizacoes rapidas

In [ ]:
plot_df = ranking_pessoas.head(20).copy()
plot_df["label"] = plot_df["cadastro"].astype(str) + " - " + plot_df["nome"].fillna("").astype(str).str.slice(0, 28)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(plot_df["label"][::-1], plot_df["qtd_pontos_apos_saida"][::-1], color="#2F80ED")
ax.set_title("Top 20 pessoas por pontos apos ultima saida PORT/S")
ax.set_xlabel("Quantidade de pontos apos saida")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

if not resumo_por_manual.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(resumo_por_manual["manual"], resumo_por_manual["qtd_pontos_apos_saida"], color="#0F766E")
    ax.set_title("Pontos apos saida por origem")
    ax.set_xlabel("registro_manual")
    ax.set_ylabel("Quantidade")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

## Exportar resultados

In [ ]:
ranking_csv = OUT_DIR / "ranking_pessoas_pontos_apos_saida_port_s.csv"
detalhes_csv = OUT_DIR / "detalhes_top_pessoas_pontos_apos_saida_port_s.csv"
resumo_manual_csv = OUT_DIR / "resumo_por_manual_pontos_apos_saida_port_s.csv"
resumo_cargo_csv = OUT_DIR / "resumo_por_cargo_pontos_apos_saida_port_s.csv"
excel_path = OUT_DIR / "analise_pontos_apos_saida_port_s.xlsx"

ranking_pessoas.to_csv(ranking_csv, index=False, encoding="utf-8-sig")
detalhes_top.to_csv(detalhes_csv, index=False, encoding="utf-8-sig")
resumo_por_manual.to_csv(resumo_manual_csv, index=False, encoding="utf-8-sig")
resumo_por_cargo.to_csv(resumo_cargo_csv, index=False, encoding="utf-8-sig")

try:
    with pd.ExcelWriter(excel_path) as writer:
        totais.to_excel(writer, sheet_name="totais", index=False)
        ranking_pessoas.to_excel(writer, sheet_name="ranking_pessoas", index=False)
        resumo_por_manual.to_excel(writer, sheet_name="resumo_manual", index=False)
        resumo_por_cargo.to_excel(writer, sheet_name="resumo_cargo", index=False)
        detalhes_top.to_excel(writer, sheet_name="detalhes_top", index=False)
    print(f"Excel gerado: {excel_path}")
except Exception as exc:
    print(f"Nao foi possivel gerar Excel ({type(exc).__name__}: {exc}). CSVs foram gerados normalmente.")

print(ranking_csv)
print(detalhes_csv)
print(resumo_manual_csv)
print(resumo_cargo_csv)

In [ ]:
con.close()